# Финальный пайплайн модели для получения submission.csv

Скачиваем файлы

In [ ]:
!wget -q --show-progress --no-check-certificate 'https://docs.google.com/uc?export=download&id=1r0mVIQfg3HYJKFIDcHwInuBPWsUBon8z' -O events.csv.gz
!wget -q --show-progress --no-check-certificate 'https://docs.google.com/uc?export=download&id=13JonFyBxaGSZJ5F_08D6tYP2uDnv8j1a' -O test.csv
!wget -q --show-progress --no-check-certificate 'https://docs.google.com/uc?export=download&id=1rwsOr06_dM4pupnO1FhOcvqWlHAM0iVe' -O train.csv
!wget -q --show-progress --no-check-certificate 'https://docs.google.com/uc?export=download&id=1DVrGiA2Nh8KvYe4os_rR2sv9ElFFgCAv' -O metric.py

events.csv.gz       100%[===================>]  11.39M  45.4MB/s    in 0.3s    
test.csv            100%[===================>] 297.28K  --.-KB/s    in 0.02s   
train.csv           100%[===================>] 693.25K  --.-KB/s    in 0.03s   
metric.py           100%[===================>]   2.37K  --.-KB/s    in 0s      


In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score
from metric import precision_at_recall
from sklearn.ensemble import RandomForestClassifier

train = pd.read_csv('/content/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('/content/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('/content/events.csv.gz', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))
train.head()

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


In [ ]:
def events_in_window(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
print(len(ev_tr), len(ev_te))

198436 89690


## Простейшие признаки

Два агрегата — сколько событий и сколько разных объявлений. Этого заведомо мало.

In [ ]:
def basic_features(ev, meta):
    g = ev.groupby('cookie_id')
    f = pd.DataFrame({
        'n_events': g.size(),
        'item_nunique': g.item_id.nunique(),
    })
    f = meta[['cookie_id']].merge(f.reset_index(), on='cookie_id', how='left')
    return f.fillna(0)

Xtr = basic_features(ev_tr, train)
Xte = basic_features(ev_te, test)
ytr = train.target.values
Xtr.head()

,cookie_id,n_events,item_nunique
0,ck_54a059eb7d3ea68b,7,4
1,ck_7e4de46eeab82974,41,19
2,ck_9320229ef6304522,36,19
3,ck_30ccd25bc1714ed9,21,10
4,ck_a77c5f05948cdeef,29,16


## Создание признаков

In [ ]:
def get_time_period(hour):
    if 0 <= hour < 6:
        return 'night'
    elif 6 <= hour < 12:
        return 'morning'
    elif 12 <= hour < 18:
        return 'afternoon'
    else:
        return 'evening'

In [ ]:
def interval_entropy(x, bins=20):
    sec = x.dt.total_seconds().dropna()
    if len(sec) < 2:
        return np.nan
    hist, _ = np.histogram(sec, bins=bins)
    p = hist / hist.sum()
    p = p[p > 0]
    return -(p * np.log2(p)).sum()

In [ ]:
def create_features(ev, meta):

    ev = ev.copy()
    # Предобработка
    ev['platform'] = (
        ev['platform']
        .fillna('')
        .str.strip()
        .str.lower()
        .replace({'iphone': 'ios'})
    )

    ev = ev.sort_values(['cookie_id', 'event_ts']).reset_index(drop=True)

    # Время
    ev['event_hour'] = ev['event_ts'].dt.hour
    ev['day_part'] = ev['event_hour'].apply(get_time_period)

    # Основная группировка по cookie
    g = ev.groupby('cookie_id', sort=False)

    # EVENT FLAGS
    event_names = [
        'item_view',
        'search_results_view',
        'photo_swipe',
        'favorite_add',
        'seller_page_view',
        'contact_message_sent',
        'login',
    ]

    for event_name in event_names:
        ev[f'is_{event_name}'] = (
            ev['event_name'] == event_name
        ).astype(int)

    event_counts = (
        ev.groupby('cookie_id')[[
            'is_item_view',
            'is_search_results_view',
            'is_photo_swipe',
            'is_favorite_add',
            'is_seller_page_view',
            'is_contact_message_sent',
            'is_login',
        ]]
        .sum()
    )

    # ACTIVITY
    activity_features = pd.DataFrame(index=g.size().index)

    activity_features['n_unique_events'] = (
        g['event_name'].nunique()
    )

    activity_features['n_unique_hours'] = (
        g['event_hour'].nunique()
    )


    # EVENTS
    event_features = pd.DataFrame(index=event_counts.index)

    event_features['n_item_view'] = (
        event_counts['is_item_view']
    )

    event_features['n_search_results_view'] = (
        event_counts['is_search_results_view']
    )

    event_features['n_photo_swipe'] = (
        event_counts['is_photo_swipe']
    )

    # Безопасное отношение двух счётчиков
    def safe_ratio(num, den):
        return (
            num
            .div(den.replace(0, np.nan))
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0)
        )

    event_features['search_results_view_favorite_add_ratio'] = (
        safe_ratio(
            event_counts['is_search_results_view'],
            event_counts['is_favorite_add']
        )
    )

    event_features['seller_page_view_contact_message_sent_ratio'] = (
        safe_ratio(
            event_counts['is_seller_page_view'],
            event_counts['is_contact_message_sent']
        )
    )

    event_features['item_view_favorite_add_ratio'] = (
        safe_ratio(
            event_counts['is_item_view'],
            event_counts['is_favorite_add']
        )
    )

    event_features['seller_page_view_login_ratio'] = (
        safe_ratio(
            event_counts['is_seller_page_view'],
            event_counts['is_login']
        )
    )


    # ITEMS
    item_features = pd.DataFrame(index=g.size().index)

    # Сколько событий приходится на каждый item
    events_per_item = (
        ev.groupby(['cookie_id', 'item_id'])
        .size()
    )

    item_features['n_events_per_item_mean'] = (
        events_per_item
        .groupby('cookie_id')
        .mean()
    )

    item_features['n_events_per_item_median'] = (
        events_per_item
        .groupby('cookie_id')
        .median()
    )

    # Сколько разных типов событий приходится на каждый item
    unique_events_per_item = (
        ev.groupby(['cookie_id', 'item_id'])['event_name']
        .nunique()
    )

    item_features['n_unique_events_per_item_mean'] = (
        unique_events_per_item
        .groupby('cookie_id')
        .mean()
    )

    item_features['n_unique_events_per_item_median'] = (
        unique_events_per_item
        .groupby('cookie_id')
        .median()
    )

    item_features['n_item_categories'] = (
        g['item_category'].nunique()
    )

    # GEOGRAPHY
    geography_features = pd.DataFrame(index=g.size().index)

    geography_features['n_unique_cities'] = (
        g['item_location'].nunique()
    )

    # Доля событий в самом популярном городе
    city_counts = (
        ev.groupby(['cookie_id', 'item_location'])
        .size()
    )

    city_shares = (
        city_counts
        / city_counts.groupby(level=0).transform('sum')
    )

    most_common_city_share = (
        city_shares
        .groupby(level=0)
        .max()
    )

    geography_features['most_common_city_share'] = (
        most_common_city_share
    )

    # SEARCH
    search_features = pd.DataFrame(index=g.size().index)

    search_features['n_unique_search_queries'] = (
        g['search_query'].nunique()
    )

    # Уникальные запросы внутри каждой категории
    queries_per_category = (
        ev.dropna(subset=['search_query', 'item_category'])
        .groupby(['cookie_id', 'item_category'])['search_query']
        .nunique()
    )

    search_features['n_unique_search_queries_in_category_mean'] = (
        queries_per_category
        .groupby('cookie_id')
        .mean()
    )

    search_features['n_unique_search_queries_in_category_median'] = (
        queries_per_category
        .groupby('cookie_id')
        .median()
    )

    # Длина запроса в словах
    query_words = (
        ev['search_query']
        .fillna('')
        .astype(str)
        .str.split()
        .str.len()
    )

    query_length = (
        pd.DataFrame({
            'cookie_id': ev['cookie_id'],
            'query_length': query_words
        })
        .groupby('cookie_id')['query_length']
    )

    search_features['search_query_length_mean'] = (
        query_length.mean()
    )

    search_features['search_query_length_median'] = (
        query_length.median()
    )

    search_features['search_page_mean'] = (
        g['search_page'].mean()
    )

    search_features['search_page_median'] = (
        g['search_page'].median()
    )

    # Доля повторных запросов
    search_features['duplicated_queries_ratio'] = (
        g['search_query']
        .apply(lambda s: s.dropna().duplicated().mean())
    )

    # TIMING
    ev['delta'] = (
        ev.groupby('cookie_id')['event_ts']
        .diff()
    )

    timing_features = pd.DataFrame(index=g.size().index)

    timing_features['gap_mean'] = (
        ev.groupby('cookie_id')['delta']
        .mean()
        .dt.total_seconds()
    )

    timing_features['gap_median'] = (
        ev.groupby('cookie_id')['delta']
        .median()
        .dt.total_seconds()
    )

    timing_features['gap_std'] = (
        ev.groupby('cookie_id')['delta']
        .std()
        .dt.total_seconds()
    )

    timing_features['entropy'] = (
        ev.groupby('cookie_id')['delta']
        .apply(interval_entropy)
    )

    # Максимальное количество событий в скользящем временном окне
    def max_events_in_window(group, window):
        if group.empty:
            return 0

        rolling_count = (
            group
            .set_index('event_ts')['event_name']
            .rolling(window)
            .count()
        )

        return rolling_count.max()

    max_1m = (
        ev.groupby('cookie_id', group_keys=False)
        .apply(lambda x: max_events_in_window(x, '1min'))
    )

    max_5m = (
        ev.groupby('cookie_id', group_keys=False)
        .apply(lambda x: max_events_in_window(x, '5min'))
    )

    max_30m = (
        ev.groupby('cookie_id', group_keys=False)
        .apply(lambda x: max_events_in_window(x, '30min'))
    )

    timing_features['max_events_1min'] = max_1m
    timing_features['max_events_5min'] = max_5m
    timing_features['max_events_30min'] = max_30m

    # USER-AGENT
    ua = (
        ev['user_agent']
        .fillna('')
        .str.lower()
    )

    ev['ua_headless_chrome'] = (
        ua.str.contains('headlesschrome', regex=False)
        .astype(int)
    )

    ev['ua_ios'] = (
        ua.str.contains(
            'iphone|ipad|ipod',
            regex=True
        )
        .astype(int)
    )

    ev['ua_safari'] = (
        ua.str.contains('safari', regex=False)
        &
        ~ua.str.contains('chrome', regex=False)
    ).astype(int)

    ev['ua_linux'] = (
        ua.str.contains('linux', regex=False)
        &
        ~ua.str.contains('android', regex=False)
    ).astype(int)

    ev['ua_android'] = (
        ua.str.contains('android', regex=False)
        .astype(int)
    )

    ua_features = (
        ev.groupby('cookie_id')[
            [
                'ua_headless_chrome',
                'ua_ios',
                'ua_safari',
                'ua_linux',
                'ua_android',
            ]
        ]
        .max()
    )

    # POINTER
    ev['prev_x'] = (
        ev.groupby('cookie_id')['pointer_x']
        .shift(1)
    )

    ev['prev_y'] = (
        ev.groupby('cookie_id')['pointer_y']
        .shift(1)
    )

    movement_mask = (
        ev['pointer_x'].notna()
        &
        ev['pointer_y'].notna()
        &
        ev['prev_x'].notna()
        &
        ev['prev_y'].notna()
    )

    ev['dx'] = ev['pointer_x'] - ev['prev_x']
    ev['dy'] = ev['pointer_y'] - ev['prev_y']

    ev['distance'] = np.sqrt(
        ev['dx'] ** 2 +
        ev['dy'] ** 2
    )

    movement_features = (
        ev.loc[movement_mask]
        .groupby('cookie_id')
        .agg(
            pointer_moves=('distance', 'size'),
            distance_median=('distance', 'median'),
        )
    )

    # DAY PART
    day_part_flags = (
        ev.assign(_val=1)
        .pivot_table(
            index='cookie_id',
            columns='day_part',
            values='_val',
            aggfunc='max',
            fill_value=0,
        )
    )

    activity_features = activity_features.join(
        day_part_flags
        .reindex(columns=[
            'night',
            'morning',
            'afternoon',
            'evening'
        ], fill_value=0)
        .rename(columns={
            'night': 'is_active_night',
            'morning': 'is_active_morning',
            'afternoon': 'is_active_afternoon',
            'evening': 'is_active_evening',
        })
    )

    # PLATFORM
    platform_flags = (
        ev.assign(_val=1)
        .pivot_table(
            index='cookie_id',
            columns='platform',
            values='_val',
            aggfunc='max',
            fill_value=0,
        )
    )

    platform_features = pd.DataFrame(
        index=g.size().index
    )

    for col in PLATFORM:
        platform = col.replace('platform_', '')

        if platform in platform_flags.columns:
            platform_features[col] = (
                platform_flags[platform]
                .reindex(platform_features.index)
                .fillna(0)
                .astype(int)
            )
        else:
            platform_features[col] = 0

    # EVENT TRANSITIONS
    ev['prev_event'] = (
        ev.groupby('cookie_id')['event_name']
        .shift(1)
    )

    ev['transition'] = (
        'transition_'
        + ev['prev_event'].fillna('')
        + '_'
        + ev['event_name'].fillna('')
    )

    transition_counts = (
        ev[
            ev['transition'].isin(TRANSITIONS)
        ]
        .groupby(['cookie_id', 'transition'])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=TRANSITIONS, fill_value=0)
    )

    # Используем доли переходов, а не абсолютные количества, чтобы признаки меньше зависели от общего n_events.
    transition_shares = (
        transition_counts
        .div(
            transition_counts.sum(axis=1)
            .replace(0, np.nan),
            axis=0
        )
        .fillna(0)
    )

    transition_features = pd.DataFrame(
        index=g.size().index
    )

    for col in TRANSITIONS:
        transition_features[col] = (
            transition_shares[col]
            .reindex(transition_features.index)
            .fillna(0)
        )

    # СОБИРАЕМ ВСЕ БЛОКИ
    f = pd.concat(
        [
            activity_features,
            event_features,
            item_features,
            geography_features,
            search_features,
            timing_features,
            ua_features,
            movement_features,
            platform_features,
            transition_features,
        ],
        axis=1
    )

    # Возвращаем строки в том же порядке cookie_id, что и в meta
    f = (
        meta[['cookie_id']]
        .drop_duplicates()
        .set_index('cookie_id')
        .join(f)
    )

    return f

In [ ]:
BASELINE = [
    'n_events',
    'item_nunique',
]

ACTIVITY = [
    'n_unique_events',
    'n_unique_hours',
    'is_active_night',
    'is_active_morning',
    'is_active_afternoon',
    'is_active_evening',
]

EVENTS = [
    'n_item_view',
    'n_search_results_view',
    'n_photo_swipe',
    'search_results_view_favorite_add_ratio',
    'seller_page_view_contact_message_sent_ratio',
    'item_view_favorite_add_ratio',
    'seller_page_view_login_ratio',
]

ITEMS = [
    'n_events_per_item_mean',
    'n_events_per_item_median',
    'n_unique_events_per_item_mean',
    'n_unique_events_per_item_median',
    'n_item_categories',
]

GEOGRAPHY = [
    'n_unique_cities',
    'most_common_city_share',
]

SEARCH = [
    'n_unique_search_queries',
    'n_unique_search_queries_in_category_mean',
    'n_unique_search_queries_in_category_median',
    'search_query_length_mean',
    'search_query_length_median',
    'search_page_mean',
    'search_page_median',
    'duplicated_queries_ratio',
]

TIMING = [
    'gap_mean',
    'gap_median',
    'gap_std',
    'entropy',
    'max_events_1min',
    'max_events_5min',
    'max_events_30min',
]

UA = [
    'ua_android',
    'ua_ios',
    'ua_safari',
    'ua_linux',
    'ua_headless_chrome',
]

POINTER = [
    'distance_median',
    'pointer_moves',
]

PLATFORM = [
    'platform_android',
    'platform_ios',
    'platform_desktop',
]

TRANSITIONS = [
    'transition_item_view_item_view',
    'transition_item_view_search_results_view',
    'transition_search_results_view_item_view',
    'transition_search_results_view_search_results_view',
]

In [ ]:
Xtr_ultim = create_features(ev_tr, train)
Xte_ultim = create_features(ev_te, test)
ytr_ultim = train.target.values

print(Xtr.shape)
print(Xtr.columns.tolist())
print(Xtr.isna().sum().sort_values(ascending=False).head(20))
print(np.isinf(Xtr.select_dtypes(include=np.number)).sum().sum())

/tmp/ipykernel_3331/2061217669.py:299: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: max_events_in_window(x, '1min'))
/tmp/ipykernel_3331/2061217669.py:304: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: max_events_in_window(x, '5min'))
/tmp/ipykernel_3331/2061217669.py:309: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is depreca

(11091, 3)
['cookie_id', 'n_events', 'item_nunique']
cookie_id       0
n_events        0
item_nunique    0
dtype: int64
0


In [ ]:
Xtr_ultim = Xtr_ultim.merge(Xtr, on=['cookie_id'])
Xte_ultim = Xte_ultim.merge(Xte, on=['cookie_id'])

## Обучение модели

In [ ]:
cols = BASELINE + ACTIVITY  + EVENTS + ITEMS + GEOGRAPHY + SEARCH + TIMING + UA + POINTER + PLATFORM + TRANSITIONS

best_xgb_model = xgb.XGBClassifier(
        max_depth=6,
        min_child_weight=3,
        learning_rate=0.025,
        n_estimators=600,

        subsample=0.8,
        colsample_bytree=0.8,

        reg_alpha=0,
        reg_lambda=1,

        random_state=0,
        eval_metric='logloss',
        n_jobs=-1
    )

## Сабмит

In [ ]:
best_xgb_model.fit(Xtr_ultim[cols], ytr_ultim)
sub = pd.DataFrame({
    'cookie_id': Xte_ultim.cookie_id,
    'score': best_xgb_model.predict_proba(Xte_ultim[cols])[:, 1],
})
assert len(sub) == len(test) and sub.score.between(0, 1).all()
sub.to_csv('submission.csv', index=False)
sub.head()

,cookie_id,score
0,ck_315fb710a0e371e7,0.000625
1,ck_a76ee3b3e3e522fd,0.090838
2,ck_94c9a4d382689e82,0.031140
3,ck_8eaf9509ad9462a0,0.001142
4,ck_9a88a5a989cb5bc6,0.007515
